In [1]:
!pip install --quiet jedi
!pip install --quiet h2o
!pip install --quiet 'thinc<8.3.6'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.4/266.4 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 80.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.14 requires confection<2.0.0,>=1.3.2, but you have confection 0.1.5 which is incompatible.
spacy 3.8.14 requires thinc<8.4.0,>=8.3.12, but you have thinc 8.3.4 which is incompatible.
weasel 1.0.0 requires confection>=1.0.0, but you have confection 0.1.5 which is incompatible.


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


In [3]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
# Initialize H2O
h2o.init(max_mem_size="2G", nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.19" 2026-04-21; OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu); OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpsxw3xvvx
  JVM stdout: /tmp/tmpsxw3xvvx/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpsxw3xvvx/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,06 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 7 days
H2O_cluster_name:,H2O_from_python_unknownUser_s5t12c
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,2 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [5]:
# Load data
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target.rename('target')

In [6]:
# Create H2OFrame
df = h2o.H2OFrame(pd.concat([X, y], axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [7]:
# Split into train/test
train, test = df.split_frame(ratios=[0.8], seed=42)

In [8]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name="california_regression"
)
aml_reg.train(x=X.columns.tolist(), y='target', training_frame=train)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_4_AutoML_1_20260729_154512


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    99                 99                          277088                 10           10           10            41            451           218.404

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.07397325656727911
RMSE: 0.2719802503257895
MAE: 0.18538668936475916
RMSLE: 0.0835524573825608
Mean Residual Deviance: 0.07397325656727911

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.20714453019161605
RMSE: 0.4551313329047079
MAE: 0.29676202261609314
RMSLE: 0.13681969786908746
Mean Residual Deviance: 0.20714453019161605

Cross-Validation Metrics Summary: 
                        mean      sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  --------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan       0            nan           nan           nan           nan           nan
loglikelihood           nan       0            nan           nan           nan           nan           nan
mae                     0.296732  0.00322323   0.299136      0.292047      0.294765      0.298175      0.299539
mean_residual_deviance  0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
mse                     0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
r2                      0.844695  0.00317222   0.840185      0.845816      0.847472      0.842682      0.847321
residual_deviance       0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
rmse                    0.455183  0.00668183   0.46659       0.452661      0.450643      0.455466      0.450555
rmsle                   0.136859  0.000486415  0.137507      0.136554      0.136472      0.136504      0.137256

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse        training_mae         training_deviance
---  -------------------  ----------  -----------------  -------------------  -------------------  -------------------
     2026-07-29 15:48:08  14.553 sec  0.0                1.1550860447424607   0.9126682923030268   1.334223770758782
     2026-07-29 15:48:08  14.683 sec  5.0                0.8099859863741191   0.6333660448978525   0.6560772981224546
     2026-07-29 15:48:08  14.821 sec  10.0               0.617830894441751    0.4722230808231802   0.38171501412669406
     2026-07-29 15:48:09  14.942 sec  15.0               0.5109753197147372   0.3784870497227261   0.26109577735757794
     2026-07-29 15:48:09  15.061 sec  20.0               0.44111368507007787  0.3156783019546691   0.19458128315610382
     2026-07-29 15:48:09  15.185 sec  25.0               0.4067439923962636   0.284500376285093    0.1654406753504517
     2026-07-29 15:48:09  15.292 sec  30.0               0.382227147884688    0.2624152441257563   0.14609759258006313
     2026-07-29 15:48:09  15.407 sec  35.0               0.3651895115355381   0.24789120218978403  0.1333633793355649
     2026-07-29 15:48:09  15.522 sec  40.0               0.34783585389151034  0.23473247741922482  0.12098978125243612
     2026-07-29 15:48:09  15.631 sec  45.0               0.3358994029907933   0.22632615955146831  0.11282840892957134
---  ---                  ---         ---                ---                  ---                  ---
     2026-07-29 15:48:10  15.951 sec  55.0               0.3176689889023286   0.21408124998465275  0.10091358651022775
     2026-07-29 15:48:10  16.137 sec  60.0              

In [9]:
# Show leaderboard
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

                         model_id      rmse       mse       mae     rmsle  \
0  GBM_4_AutoML_1_20260729_154512  0.455131  0.207145  0.296762  0.136820   
1  GBM_2_AutoML_1_20260729_154512  0.455992  0.207929  0.300417  0.137703   
2  GBM_3_AutoML_1_20260729_154512  0.458074  0.209832  0.301015  0.137618   
3  GBM_1_AutoML_1_20260729_154512  0.459199  0.210863  0.302926  0.138295   
4  GBM_5_AutoML_1_20260729_154512  0.460024  0.211622  0.305412  0.139356   

   mean_residual_deviance  
0                0.207145  
1                0.207929  
2                0.209832  
3                0.210863  
4                0.211622  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [10]:
# Evaluate on test
perf_reg = aml_reg.leader.model_performance(test)
print(f"H2O Regression R²: {perf_reg.r2():.4f}")
print(f"H2O Regression RMSE: {perf_reg.rmse():.4f}")

H2O Regression R²: 0.8523
H2O Regression RMSE: 0.4417


In [11]:
from sklearn.datasets import load_breast_cancer

In [12]:
# Load and prepare dataset
data_cls = load_breast_cancer(as_frame=True)
Xc = data_cls.data
yc = data_cls.target.rename('target')

In [13]:
df_cls = h2o.H2OFrame(pd.concat([Xc, yc], axis=1))
train_cls, test_cls = df_cls.split_frame(ratios=[0.7], seed=42)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [14]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    balance_classes=True,
    project_name="breast_cancer_classification"
)
aml_cls.train(x=Xc.columns.tolist(), y='target', training_frame=train_cls)

AutoML progress: |
15:50:15.911: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

██
15:50:22.985: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
15:50:24.353: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
15:50:29.365: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

██
15:50:35.703: _response param, We have 

key,value
Stacking strategy,cross_validation
Number of base models (used / total),4/5
# GBM base models (used / total),1/1
# XGBoost base models (used / total),0/1
# DRF base models (used / total),2/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [15]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                                            model_id      rmse       mse  \
0  StackedEnsemble_BestOfFamily_1_AutoML_2_202607...  0.182021  0.033132   
1  StackedEnsemble_AllModels_1_AutoML_2_20260729_...  0.184126  0.033903   
2       GBM_grid_1_AutoML_2_20260729_155015_model_22  0.185965  0.034583   
3       GBM_grid_1_AutoML_2_20260729_155015_model_11  0.186716  0.034863   
4       GBM_grid_1_AutoML_2_20260729_155015_model_20  0.188528  0.035543   

        mae     rmsle  mean_residual_deviance  
0  0.102690  0.130640                0.033132  
1  0.097524  0.132021                0.033903  
2  0.096905  0.131893                0.034583  
3  0.089201  0.134120                0.034863  
4  0.090711  0.133354                0.035543  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [16]:
# Evaluate on test
perf_cls = aml_cls.leader.model_performance(test_cls)
print(f"H2O Classification: {perf_cls}")

H2O Classification: ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 0.037043105199522024
RMSE: 0.19246585463276863
MAE: 0.1050351571585801
RMSLE: 0.1322812239323879
Mean Residual Deviance: 0.037043105199522024
R^2: 0.8355508273586023
Null degrees of freedom: 177
Residual degrees of freedom: 173
Null deviance: 40.43221198186829
Residual deviance: 6.593672725514921
AIC: -69.48768312783018


## 7. Exercise: Custom Dataset AutoML

**Task:** Apply H2O AutoML to a custom dataset.

1. Load any tabular dataset (CSV or from `sklearn.datasets`).
2. Decide whether it's a regression or classification task.
3. Initialize H2O and convert to `H2OFrame`.
4. Split into appropriate train/test ratios.
5. Run `H2OAutoML` with:
   - `max_runtime_secs=300`
   - `max_models=15`
   - `nfolds=5`
   - `balance_classes=True` (if classification)
6. Display the leaderboard and evaluate on the test set using relevant metrics.
7. Save the leaderboard to a pandas DataFrame and export it as `leaderboard.csv`

In [17]:
from sklearn.datasets import load_wine

# Load custom dataset (classification: 3 wine classes)
data_wine = load_wine(as_frame=True)
Xw = data_wine.data
yw = data_wine.target.rename('target')

In [18]:
h2o.init(max_mem_size="2G", nthreads=-1)

df_wine = h2o.H2OFrame(pd.concat([Xw, yw], axis=1))
df_wine['target'] = df_wine['target'].asfactor()   # classification ke liye target ko factor banana zaroori hai

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,11 mins 10 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 7 days
H2O_cluster_name:,H2O_from_python_unknownUser_s5t12c
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.823 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [19]:
train_wine, test_wine = df_wine.split_frame(ratios=[0.8], seed=42)

In [20]:
aml_wine = H2OAutoML(
    max_runtime_secs=300,
    max_models=15,
    nfolds=5,
    balance_classes=True,
    seed=42,
    project_name="wine_classification"
)
aml_wine.train(x=Xw.columns.tolist(), y='target', training_frame=train_wine)

AutoML progress: |███
15:56:37.632: _min_rows param, The dataset size is too small to split for min_rows=100.0: must have at least 200.0 (weighted) rows, but have only 146.0.

████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OXGBoostEstimator : XGBoost
Model Key: XGBoost_grid_1_AutoML_3_20260729_155627_model_1


Model Summary: 
    number_of_trees
--  -----------------
    47

ModelMetricsMultinomial: xgboost
** Reported on train data. **

MSE: 0.00020003735022558136
RMSE: 0.014143456091973466
LogLoss: 0.009331187570041933
Mean Per-Class Error: 0.0
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0    1    2    Error    Rate
---  ---  ---  -------  -------
49   0    0    0        0 / 49
0    58   0    0        0 / 58
0    0    39   0        0 / 39
49   58   39   0        0 / 146

Top-3 Hit Ratios: 
k    hit_ratio
---  -----------
1    1
2    1
3    1

ModelMetricsMultinomial: xgboost
** Reported on cross-validation data. **

MSE: 0.017083353053440265
RMSE: 0.13070330161644833
LogLoss: 0.06874893608409434
Mean Per-Class Error: 0.005747126436781609
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0    1    2    Error       Rate
---  ---  ---  ----------  -------
49   0    0    0           0 / 49
1    57   0    0.0172414   1 / 58
0    0    39   0           0 / 39
50   57   39   0.00684932  1 / 146

Top-3 Hit Ratios: 
k    hit_ratio
---  -----------
1    0.993151
2    1
3    1

Cross-Validation Metrics Summary: 
                         mean        sd         cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
-----------------------  ----------  ---------  ------------  ------------  ------------  ------------  ------------
accuracy                 0.993103    0.0154212  1             0.965517      1             1             1
aic                      nan         0          nan           nan           nan           nan           nan
auc                      nan         0          nan           nan           nan           nan           nan
err                      0.00689655  0.0154212  0             0.0344828     0             0             0
err_count                0.2         0.447214   0             1             0             0             0
loglikelihood            nan         0          nan           nan           nan           nan           nan
logloss                  0.0688845   0.045691   0.0490866     0.144061      0.0796105     0.0339311     0.0377333
max_per_class_error      0.0166667   0.0372678  0             0.0833333     0             0             0
mean_per_class_accuracy  0.994444    0.0124226  1             0.972222      1             1             1
mean_per_class_error     0.00555556  0.0124226  0             0.0277778     0             0             0
mse                      0.0171321   0.0144227  0.0100196     0.0404117     0.0216853     0.00820941    0.00533443
pr_auc                   nan         0          nan           nan           nan           nan           nan
r2                       0.97106     0.0251331  0.983176      0.929781      0.964793      0.986672      0.990882
rmse                     0.122405    0.0518292  0.100098      0.201026      0.147259      0.0906058     0.0730372

Scoring History: 
    timestamp            duration    number_of_trees    training_rmse    training_logloss    training_classification_error    training_auc    training_pr_auc
--  -------------------  ----------  -----------------  ---------------  ------------------  -------------------

In [21]:
# Show leaderboard
df_leader_wine = aml_wine.leaderboard.as_data_frame()
print(df_leader_wine.head())

# Evaluate on test
perf_wine = aml_wine.leader.model_performance(test_wine)
print(f"H2O Wine Classification: {perf_wine}")

                                            model_id  mean_per_class_error  \
0    XGBoost_grid_1_AutoML_3_20260729_155627_model_1              0.005747   
1  StackedEnsemble_BestOfFamily_1_AutoML_3_202607...              0.005747   
2                     GBM_3_AutoML_3_20260729_155627              0.011494   
3        GBM_grid_1_AutoML_3_20260729_155627_model_1              0.011494   
4                     GBM_2_AutoML_3_20260729_155627              0.011494   

    logloss      rmse       mse  
0  0.068749  0.130703  0.017083  
1  0.054144  0.113538  0.012891  
2  0.037751  0.108852  0.011849  
3  0.059132  0.123506  0.015254  
4  0.040875  0.109542  0.011999  
H2O Wine Classification: ModelMetricsMultinomial: xgboost
** Reported on test data. **

MSE: 0.006538160240691315
RMSE: 0.0808588909192509
LogLoss: 0.04197425012913035
Mean Per-Class Error: 0.0
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the

/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [22]:
df_leader_wine.to_csv("leaderboard.csv", index=False)
print("Saved leaderboard.csv")

Saved leaderboard.csv
